# 02 · Why does it spread? (Act 1b: traits, natural notes, manipulation check)

**Question.** Which rhetorical traits make a shortcut contagious, and do LLM-written notes spread like templates?

**What it runs.** Natural notes (6 voices), LLM manipulation check, exp_002 (2⁶ trait factorial), exp_006 (natural vs template).

**Runtime:** CPU · ~5-6 h (exp_002 is the long one)

Every command is resumable: if Colab disconnects, re-run the same cell. Finished episodes are skipped and
cached calls are replayed without new API calls.

In [ ]:
# Setup 1/2 - get the code: clones the repo (or unzips /content/outbreak.zip if you uploaded it) and installs it
import pathlib, subprocess
REPO_URL = "https://github.com/albertogomezsoteres/outbreak.git"
if not pathlib.Path("/content/outbreak").exists():
    if pathlib.Path("/content/outbreak.zip").exists():
        subprocess.run(["unzip", "-q", "/content/outbreak.zip", "-d", "/content"], check=True)
    else:
        subprocess.run(["git", "clone", "-q", REPO_URL, "/content/outbreak"], check=True)
%cd /content/outbreak
!pip -q install -e ".[api]"
# The running kernel does not see a package installed a moment ago: put src/ on the path and drop any stale
# `outbreak` module (Colab has /content on sys.path, where the repo folder itself is also called outbreak).
import sys, importlib
sys.path.insert(0, "/content/outbreak/src")
for m in [m for m in sys.modules if m == "outbreak" or m.startswith("outbreak.")]:
    del sys.modules[m]
importlib.invalidate_caches()

In [ ]:
# Setup 2/2 - Drive for results, local cache (restored from Drive), API key from Colab Secrets
from outbreak.colab import setup, save_cache, show_report, show_figure
setup()

## 1. Natural notes, then check that raters perceive the designed traits

In [ ]:
NAT = "outputs/results/natural_notes"
!outbreak natural-notes --model openai/gpt-5-mini --n 48 --out-dir {NAT}
!outbreak manipulation-check --notes data/processed/notes_M2.jsonl data/processed/notes_M3.jsonl {NAT}/natural_notes_M2.jsonl {NAT}/natural_notes_M3.jsonl --model openai/gpt-5-mini --out outputs/results/manipulation_check
import pandas as pd; display(pd.read_csv("outputs/results/manipulation_check/agreement.csv"))

## 2. Trait factorial (exp_002): 192 notes per shortcut, randomly assigned to scenario draws

In [ ]:
!outbreak run experiments/single_experiments/02_trait_factorial.yaml --run-id exp_002_main
!outbreak analyze outputs/results/exp_002_main/episodes.jsonl
show_figure("outputs/results/exp_002_main/analysis/fig_trait_heatmap.png")
display(pd.read_csv("outputs/results/exp_002_main/analysis/h3_trait_risk_differences.csv"))

## 3. Natural vs template notes (exp_006)

In [ ]:
!outbreak run experiments/single_experiments/06_natural_notes.yaml --run-id exp_006_main -o 'notes.natural_dir={NAT}'
!outbreak analyze outputs/results/exp_006_main/episodes.jsonl
show_report("outputs/results/exp_006_main/analysis/report.md", sections=("H1 - shortcut", "Natural"))

In [ ]:
save_cache()